### Imports

In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,List, Annotated
from langchain_core.messages import BaseMessage, HumanMessage,AIMessage,SystemMessage
from langgraph.graph.message import add_messages

### Graph State

In [2]:
class AgentState(TypedDict):
    messages:Annotated[List[BaseMessage],add_messages]
    category:str
    is_angry:str
    is_resolved:str
    generated_response: str

### Classification Node

In [3]:
from pydantic import BaseModel,Field
from typing import Annotated,List,Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

c:\Users\LENOVO\Desktop\Customer Support Bot\customerbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
# Pydantic Model
class ClassificationOutput(BaseModel):
    category:Literal["billing", "faq" , "technical", "general"] = Field(description="The exact category of the user's query.")
    is_angry: str = Field(description="Return the exact string 'True' ONLY if the user is using profanity, expressing extreme frustration, or explicitly demanding a human. Otherwise return the string 'False'.")

In [5]:
# Intialize LLMs
llm = ChatGroq(model = "llama-3.1-8b-instant")
llm_classifier = llm.with_structured_output(ClassificationOutput)

In [6]:
system_prompt = """You are an expert customer support triage agent.
Your job is to analyze the user's message and classify it into one of these specific categories:
- billing: Queries about invoices, payments, refunds, or subscription plans.
- faq: General company policy, operating hours, or basic non-technical questions.
- technical: Technical troubleshooting, bug reports, system errors, or API issues.
- general: Any standard conversational input (like "hello") or questions that do not fit the above.

You must also evaluate the user's emotional state. Flag 'is_angry' as True if the user is highly agitated or demands human escalation."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

# Combine into a single runnable chain
classification_chain = prompt | llm_classifier

In [7]:
# Node Function
def classify_query_node(state : AgentState):
    user_query = state["messages"][-1].content
    result = classification_chain.invoke({"question": user_query})

    # Convert the string "True"/"False" safely into a Python boolean
    is_angry_bool = (result.is_angry.lower() == "true")

    print(f"Decision -> Category: {result.category} | Angry: {is_angry_bool}")
    
    # Return the dictionary to update our AgentState
    return {
        "category": result.category,
        "is_angry": is_angry_bool
    }



### RAG 

In [8]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
# step 1 : Document Loader
loader = DirectoryLoader(
    path = './data',
    glob = '**/*.txt',
    loader_cls = TextLoader
)

docs = loader.load()

# Print how many files were successfully loaded to verify
print(f"Successfully loaded {len(docs)} text documents.")

Successfully loaded 3 text documents.


In [10]:
# step 2 : Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 600, 
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

split_docs = text_splitter.split_documents(docs)

# Print the results clearly to see the overlap
print(f"Total Chunks Created: {len(split_docs)}\n")
print("="*50)

# Print the first 3 chunks to inspect them
for i in range(8):
    print(f"--- CHUNK {i + 1} ---")
    print(split_docs[i].page_content)
    print("\n")




Total Chunks Created: 14

--- CHUNK 1 ---
# Frequently Asked Questions (FAQ)

## Shipping and Delivery
**Q: How long does shipping take?**
A: Standard shipping typically takes 3-5 business days. Expedited shipping is available for an additional fee and takes 1-2 business days. Orders placed after 2 PM EST will be processed the next business day.

**Q: Do you ship internationally?**
A: Yes, we ship to most countries worldwide. International shipping usually takes 7-14 business days, depending on customs processing in your country. Please note that customers are responsible for any local customs duties or import taxes.


--- CHUNK 2 ---
**Q: How can I track my order?**
A: Once your order ships, you will receive an email with a tracking number and a link to track your package on our carrier's website. You can also track your order directly in your "Account Dashboard".

**Q: What happens if my package is lost or stolen?**
A: If your tracking information shows "Delivered" but you have not r

In [11]:
# Step 3 : Embedding Creation
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Test
sample_query = "What is your refund policy?"
vector = embedding.embed_query(sample_query)
print(f"Successfully generated a vector with {len(vector)} dimensions!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1585.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully generated a vector with 384 dimensions!


In [20]:
# Step 4: Create Vector Stores 
vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embedding
)

# Save the database locally to a folder named "faiss_index"
vectorstore.save_local("faiss_index")

print("Successfully created and saved the FAISS database!")




Successfully created and saved the FAISS database!


In [17]:
# Step 5: Retriver
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",  
    search_kwargs={
        "k": 3, # Find 3 Relevant chunk
        "score_threshold": 0.4 # Bring me up to 3 chunks, but only if they have a 50% or higher similarity match!
        })

# Testing
query = "Process for refund?"
retrieved_docs = retriever.invoke(query)

print(f"\nFound {len(retrieved_docs)} relevant chunks for your query:\n")
print("="*50)
for i, doc in enumerate(retrieved_docs):
    print(f"--- MATCH {i + 1} ---")
    print(doc.page_content)
    print("="*50)


Found 2 relevant chunks for your query:

--- MATCH 1 ---
## Process for Refunds
To initiate a refund, please follow these steps:
1. Log in to your account and navigate to "Order History".
2. Select the order and click "Request Refund".
3. Select the reason for the return from the dropdown menu.
4. Print the provided pre-paid return label and attach it to the package.
5. Drop off the package at any authorized shipping center.

Once we receive and inspect the returned item, it takes 3-5 business days to process the refund to the original payment method.
--- MATCH 2 ---
## Late or Missing Refunds
If you havenâ€™t received a refund within 7 business days of our confirmation email, first check your bank account again. Then contact your credit card company, it may take some time before your refund is officially posted. If youâ€™ve done all of this and you still have not received your refund yet, please contact our support team.


In [14]:
# LLM Generation part
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatGroq(model = "llama-3.1-8b-instant")

system_prompt1 = (
    "You are a helpful customer support assistant for an e-commerce company. "
    "Use the following pieces of retrieved context to answer the user's question. "
    "If the answer is not in the context, just say 'I don't have that information'. "
    "Do not make up answers. Keep your response clear and concise."
    "\n\n"
    "Context: {context}"
)

# Combine the system instructions with the user's question
prompt1 = ChatPromptTemplate.from_messages([
    ("system", system_prompt1),
    ("human", "{input}"),
])

# 3. Build the RAG Chains
# Step A: A chain that figures out how to stuff your text chunks into the {context} variable
qa_chain = create_stuff_documents_chain(llm, prompt1)

# Step B: The final chain that links your FAISS retriever to the QA chain
# (This assumes your retriever from the previous step is named 'smart_retriever')
rag_chain = create_retrieval_chain(retriever, qa_chain)

# --- FINAL EXECUTION ---
print("\nAsking the LLM...")
user_question = "How long does order take to delier?"

# Run the entire pipeline!
response = rag_chain.invoke({"input": user_question})

print("\n" + "="*50)
print("USER QUESTION:", user_question)
print("-" * 50)
print("AI ANSWER:", response["answer"])
print("="*50)



No relevant docs were retrieved using the relevance score threshold 0.4



Asking the LLM...

USER QUESTION: How long does order take to delier?
--------------------------------------------------
AI ANSWER: Our delivery times vary depending on the shipping method chosen at checkout. Here are some general guidelines:

- For standard shipping, orders typically take 3-7 business days to arrive after shipping.
- For expedited shipping, orders usually arrive within 2-3 business days.
- For express shipping, orders are typically delivered within 1-2 business days.

Please note that delivery times may be affected by factors such as location, weather, and high-order volume. You can check the estimated delivery time for your order by logging into your account and viewing your order details.


### RAG NODE

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.load_local("faiss_index", embedding, allow_dangerous_deserialization=True)


retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",  
    search_kwargs={"k": 3, "score_threshold": 0.5} 
)


llm = ChatGroq(model="llama-3.1-8b-instant")

system_prompt = (
    "You are a helpful customer support assistant for an e-commerce company. "
    "Use the following pieces of retrieved context to answer the user's question. "
    "If the answer is not in the context, just say 'I don't have that information'. "
    "Do not make up answers. Keep your response clear and concise."
    "\n\nContext: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)


def rag_node(state: AgentState):
    print("---RUNNING RAG NODE---")
    
    # Get the latest message from the graph state
    user_question = state["messages"][-1].content
    
    # Run your exact chain
    response = rag_chain.invoke({"input": user_question})
    
    # Return the answer to update the graph state
    return {
        "generated_response": response["answer"],
        "messages": [AIMessage(content=response["answer"])]
        }

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1526.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### LLM Direct Node

In [22]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

direct_llm = ChatGroq(model="llama-3.1-8b-instant")

direct_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a highly skilled technical and general support agent to resolve technical problem in any Purchased product  "
        "Answer the user's question clearly and concisely. "
        "If they are just saying hello, greet them politely. "
        "Do not make up specific company policies."
    )),
    ("human", "{question}")
])


direct_chain = direct_prompt | direct_llm

def llm_direct_node(state: AgentState): 
    # Get the user's question from the state
    user_question = state["messages"][-1].content
    
    # Generate the answer directly
    response = direct_chain.invoke({"question": user_question})
    
    # Update the state with the new answer
    return {
        "generated_response": response.content,
        "messages": [AIMessage(content=response["answer"])]
        }

### Evaluator Node

In [17]:
from pydantic import BaseModel,Field
from typing import List,Annotated

# Pydantic Schema
class EvaluationOutput(BaseModel):
    is_resolved: str = Field( description="Return the exact string 'True' if the response answers the question successfully. Return the string 'False' if the response says 'I don't have that information' or is completely unhelpful.")

evaluator_llm = ChatGroq(model="llama-3.1-8b-instant").with_structured_output(EvaluationOutput)

evaluator_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a quality assurance bot. Did the 'Generated Response' "
        "successfully answer the 'User Query'? Respond carefully."
    )),
    ("human", "User Query: {query}\n\nGenerated Response: {response}")
])

eval_chain = evaluator_prompt | evaluator_llm

#  The Evaluator  Node Function
def resolution_evaluator_node(state: AgentState):
    user_query = state["messages"][-1].content
    draft_response = state["generated_response"]
    
  
    result = eval_chain.invoke({"query": user_query, "response": draft_response})
    # Convert the string "True"/"False" safely into a Python boolean
    is_resolved_bool = (result.is_resolved.lower() == "true")
    
    print(f"Is Resolved? {is_resolved_bool}")
    
    # Update the boolean flag in the state
    return {"is_resolved": is_resolved_bool}


### Human Handoff Node

In [23]:
def human_agent_node(state: AgentState):
    
    handoff_message = "I am unable to resolve this request completely. I'm transferring you to a human agent who can better assist you. Please hold."
    
    return {""
    "generated_response": handoff_message,
    "messages": [AIMessage(content=handoff_message)]
    }

### Routing Nodes

In [19]:
def route_after_classification(state: AgentState):
    """Routes the query based on category and sentiment."""
    # If the user is extremely angry, bypass the AI and go straight to a human
    if state.get("is_angry"):
        return "human_agent_node"
    
    category = state.get("category")
    
    # Route to your unified Groq/FAISS RAG node for billing or FAQ
    if category in ["billing", "faq"]:
        return "rag_node"
    # Route to the direct LLM node for technical or general questions
    else:
        return "llm_direct_node"

def route_after_evaluation(state: AgentState):
    """Routes based on whether the AI successfully answered the question."""
    # If the evaluator marked it as unresolvable, send to human
    if state.get("is_resolved") is False:
        return "human_agent_node"
    
    # If resolved successfully, we are done!
    return "end"

In [24]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# 1. Initialize the Graph with your AgentState schema
workflow = StateGraph(AgentState)

# 2. Add  nodes
workflow.add_node("classify_query_node", classify_query_node)
workflow.add_node("rag_node", rag_node)
workflow.add_node("llm_direct_node", llm_direct_node)
workflow.add_node("resolution_evaluator_node", resolution_evaluator_node)
workflow.add_node("human_agent_node", human_agent_node)

# 3. Add Edges

# Set the Entry Point (START -> Classifier)
workflow.add_edge(START, "classify_query_node")

# Add the First Conditional Split (After Classification)
workflow.add_conditional_edges(
    "classify_query_node",
    route_after_classification,
    {
        "rag_node": "rag_node",
        "llm_direct_node": "llm_direct_node",
        "human_agent_node": "human_agent_node"
    }
)

# Connect the Generation Nodes to the Evaluator
# No matter which bot generated the answer, we must evaluate it
workflow.add_edge("rag_node", "resolution_evaluator_node")
workflow.add_edge("llm_direct_node", "resolution_evaluator_node")

#  Add the Second Conditional Split (After Evaluation)
workflow.add_conditional_edges(
    "resolution_evaluator_node",
    route_after_evaluation,
    {
        "end": END,
        "human_agent_node": "human_agent_node"
    }
)

#  Connect the Human Agent to END
workflow.add_edge("human_agent_node", END)

# 8. Compile the workflow
app = workflow.compile(checkpointer=memory)

In [21]:
from langchain_core.messages import HumanMessage

if __name__ == "__main__":
    print("\n" + "="*50)
    print("STARTING CHATBOT TEST")
    print("="*50)
    
    # Create a mock user question
    user_input = "How do I reset my password?"
    
    # Initialize the state with the user's message
    initial_state = {
        "messages": [HumanMessage(content=user_input)]
    }
    
    # Run the graph! 
    # This will flow through the classifier -> RAG/LLM -> Evaluator
    final_state = app.invoke(initial_state)
    
    print("\n" + "="*50)
    print("FINAL CHATBOT RESPONSE:")
    print(final_state["generated_response"])
    print("="*50)


STARTING CHATBOT TEST
Decision -> Category: general | Angry: False
Is Resolved? True

FINAL CHATBOT RESPONSE:
To reset your password, you can follow these general steps:

1. Go to the login page of the product or service you're trying to access.
2. Click on the "Forgot Password" or "Reset Password" link.
3. Enter the email address or username associated with your account.
4. You may be asked to answer a security question or provide additional information to verify your identity.
5. Click on the "Submit" or "Reset" button.
6. You will receive an email with a password reset link or instructions.
7. Click on the link or follow the instructions in the email to reset your password.
8. Create a new password and confirm it.
9. Log in to your account with your new password.

If you're still having trouble, you can also try:

- Checking your email inbox and spam folder for the password reset email.
- Contacting the product or service's customer support for assistance.
- Trying a different brow